In [ ]:
pip install torch torchvision transformers pillow

In [ ]:
from transformers import CLIPProcessor, CLIPModel
import torch
from PIL import Image

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [ ]:
image = Image.open("Cricet-Bhumrah.jpg")

In [ ]:
captions=[
    "Virat Kohli playing a cover drive shot",
    "Jasprit Bumrah bowling a fast delivery in a cricket match",
    "MS Dhoni finishing a match with a six",
    "Rohit Sharma batting in a cricket match",
    "Jasprit Bumrah batting with a cricket bat",
    "A baseball pitcher throwing a fast pitch",
    "A cricket fast bowler delivering the ball in a stadium",
    "A tennis player serving the ball"
]

In [ ]:
inputs = processor(
    text=captions,
    images=image,
    return_tensors="pt",
    padding=True
)

outputs = model(**inputs)

In [ ]:
logits_per_image = outputs.logits_per_image
probs = logits_per_image.softmax(dim=1)

In [ ]:
probs = probs[0]   # remove batch dimension

for caption, score in zip(captions, probs):
    print(f"{caption} --> {score.item():.4f}")

Virat Kohli playing a cover drive shot --> 0.0182
Jasprit Bumrah bowling a fast delivery in a cricket match --> 0.3763
MS Dhoni finishing a match with a six --> 0.0037
Rohit Sharma batting in a cricket match --> 0.0370
Jasprit Bumrah batting with a cricket bat --> 0.0819
A baseball pitcher throwing a fast pitch --> 0.0000
A cricket fast bowler delivering the ball in a stadium --> 0.4826
A tennis player serving the ball --> 0.0003


1. A cricket fast bowler delivering the ball in a stadium → 0.4826
✅ (highest)

2. Jasprit Bumrah bowling a fast delivery in a cricket match → 0.3763 ✅ (second)

“Both captions match — but the generic one matches slightly better.”

CLIP is trained on huge image–text pairs from the internet.
It is better at:

✅ general visual descriptions

❌ specific person-identity linking (unless very famous & clear)


CLIP looks at:
pose,
motion,
uniform,
context &
objects

It does NOT verify identity reliably.

In [ ]:
best_index = probs.argmax()
print("Best Caption:", captions[best_index])

Best Caption: A cricket fast bowler delivering the ball in a stadium
